In [1]:

import pandas as pd
from pathlib import Path

# Adjust these paths if your files are stored elsewhere
PROJECT_DIR = Path("..")
KAGGLE_PATH = PROJECT_DIR / "data/raw/ir_train.csv"
NETWORK_PATH = PROJECT_DIR / "data/processed/railway_network_edges_flagged.csv"

# Read only the train-number column from Kaggle
kaggle_ids = pd.read_csv(
    KAGGLE_PATH,
    usecols=["train_number"],
    dtype={"train_number": "string"}
)

# Read train numbers from the prepared railway network
network_ids = pd.read_csv(
    NETWORK_PATH,
    usecols=["Train No"],
    dtype={"Train No": "string"}
)

# Normalize whitespace
kaggle_ids["train_number"] = kaggle_ids["train_number"].str.strip()
network_ids["Train No"] = network_ids["Train No"].str.strip()

kaggle_trains = set(kaggle_ids["train_number"].dropna().unique())
timetable_trains = set(network_ids["Train No"].dropna().unique())

matched = kaggle_trains & timetable_trains

print("Unique Kaggle train numbers:", len(kaggle_trains))
print("Unique timetable train numbers:", len(timetable_trains))
print("Matching train numbers:", len(matched))
print(
    "Kaggle train-number match rate:",
    round(100 * len(matched) / max(len(kaggle_trains), 1), 2),
    "%"
)

print("\nSample matching IDs:", sorted(matched)[:20])


Unique Kaggle train numbers: 6293
Unique timetable train numbers: 11112
Matching train numbers: 1820
Kaggle train-number match rate: 28.92 %

Sample matching IDs: ['11101', '11102', '11103', '11104', '11107', '11108', '11109', '11110', '11111', '11112', '11123', '11124', '11125', '11126', '11201', '11202', '11203', '11204', '11205', '11206']


In [2]:

# Find a few matching train IDs
sample_ids = sorted(matched)[:10]

print("Sample matched train IDs:", sample_ids)

# Inspect timetable routes for these IDs
sample_routes = pd.read_csv(
    NETWORK_PATH,
    dtype={
        "Train No": "string",
        "Station Code": "string",
        "Next Station Code": "string",
    }
)

sample_routes["Train No"] = sample_routes["Train No"].str.strip()

display(
    sample_routes[
        sample_routes["Train No"].isin(sample_ids)
    ][
        ["Train No", "Station Code", "Next Station Code",
         "Segment Distance", "distance_review"]
    ].head(30)
)

# Inspect Kaggle records for the same IDs
kaggle_sample = pd.read_csv(
    KAGGLE_PATH,
    dtype={"train_number": "string"},
    nrows=0
)

print("\nKaggle columns:")
print(kaggle_sample.columns.tolist())


Sample matched train IDs: ['11101', '11102', '11103', '11104', '11107', '11108', '11109', '11110', '11111', '11112']


,Train No,Station Code,Next Station Code,Segment Distance,distance_review
2688,11101,PUNE,CCH,16.0,False
2689,11101,CCH,LNL,47.0,False
2690,11101,LNL,PNVL,57.0,False
2691,11101,PNVL,BSR,65.0,False
2692,11101,BSR,BOR,51.0,False
2693,11101,BOR,VAPI,71.0,False
2694,11101,VAPI,BL,25.0,False
2695,11101,BL,ST,68.0,False
2696,11101,ST,BH,59.0,False
2697,11101,BH,BRC,70.0,False



Kaggle columns:
['journey_id', 'train_number', 'train_type', 'departure_date', 'year', 'month', 'day_of_week', 'departure_hour', 'is_weekend', 'is_night_departure', 'is_peak_hour', 'is_festival_season', 'season', 'zone', 'zone_abbr', 'source_station_category', 'destination_station_category', 'distance_km', 'num_scheduled_stops', 'scheduled_travel_hours', 'track_doubled', 'is_hdn_route', 'traction_type', 'is_electrified', 'psr_count', 'is_circular_route', 'is_monsoon_season', 'is_fog_risk', 'fog_risk_score', 'zone_fog_index', 'zone_congestion_index', 'season_severity_score', 'loco_age_years', 'coach_age_years', 'has_lhb_coaches', 'is_rake_shared', 'maintenance_score', 'seat_utilisation_pct', 'is_overloaded', 'late_incoming_rake', 'is_special_train', 'route_historical_ontime_pct', 'primary_delay_cause', 'delay_minutes', 'is_delayed']


In [3]:

# Display all Kaggle column names, one per line
kaggle_header = pd.read_csv(
    KAGGLE_PATH,
    nrows=0
)

for column in kaggle_header.columns:
    print(column)


journey_id
train_number
train_type
departure_date
year
month
day_of_week
departure_hour
is_weekend
is_night_departure
is_peak_hour
is_festival_season
season
zone
zone_abbr
source_station_category
destination_station_category
distance_km
num_scheduled_stops
scheduled_travel_hours
track_doubled
is_hdn_route
traction_type
is_electrified
psr_count
is_circular_route
is_monsoon_season
is_fog_risk
fog_risk_score
zone_fog_index
zone_congestion_index
season_severity_score
loco_age_years
coach_age_years
has_lhb_coaches
is_rake_shared
maintenance_score
seat_utilisation_pct
is_overloaded
late_incoming_rake
is_special_train
route_historical_ontime_pct
primary_delay_cause
delay_minutes
is_delayed


In [4]:

# Load a small sample of rows for matching train IDs
sample_ids = sorted(matched)[:10]

kaggle_check = pd.read_csv(
    KAGGLE_PATH,
    dtype={"train_number": "string"}
)

kaggle_check["train_number"] = (
    kaggle_check["train_number"].str.strip()
)

display(
    kaggle_check[
        kaggle_check["train_number"].isin(sample_ids)
    ][
        ["journey_id", "train_number", "departure_date",
         "source_station_category", "destination_station_category",
         "distance_km", "delay_minutes"]
    ].head(20)
)


,journey_id,train_number,departure_date,source_station_category,destination_station_category,distance_km,delay_minutes
197,IR01177612,11111,2019-06-22,D,C,529,123
350,IR01247335,11103,2019-11-01,E,A,914,13
507,IR00079185,11110,2019-03-08,E,B,512,95
578,IR00658405,11108,2024-07-05,A1,C,466,224
701,IR01753057,11103,2019-08-23,C,E,454,146
1625,IR01254127,11109,2021-06-12,B,C,622,185
2026,IR00177084,11101,2022-07-08,E,B,466,124
2051,IR00025856,11101,2024-09-10,B,A,598,1
2293,IR00271697,11112,2022-09-20,D,A1,851,132
2726,IR00291130,11109,2018-04-12,A,A,819,11


In [6]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"

CLEAN_TIMETABLE_PATH = (
    DATA_DIR / "processed" / "railway_timetable_clean.csv"
)

timetable_clean = pd.read_csv(
    CLEAN_TIMETABLE_PATH,
    dtype={
        "Train No": "string",
        "Station Code": "string",
    },
    low_memory=False,
)

timetable_clean["SEQ"] = pd.to_numeric(
    timetable_clean["SEQ"], errors="coerce"
)
timetable_clean["Distance"] = pd.to_numeric(
    timetable_clean["Distance"], errors="coerce"
)

print("Timetable loaded:", timetable_clean.shape)
display(timetable_clean.head())


Timetable loaded: (186114, 12)


,Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Source Station Name,Destination Station,Destination Station Name
0,107,SWV-MAO-VLNK,1.0,SWV,SAWANTWADI R,00:00:00,10:25:00,0.0,SWV,SAWANTWADI ROAD,MAO,MADGOAN JN.
1,107,SWV-MAO-VLNK,2.0,THVM,THIVIM,11:06:00,11:08:00,32.0,SWV,SAWANTWADI ROAD,MAO,MADGOAN JN.
2,107,SWV-MAO-VLNK,3.0,KRMI,KARMALI,11:28:00,11:30:00,49.0,SWV,SAWANTWADI ROAD,MAO,MADGOAN JN.
3,107,SWV-MAO-VLNK,4.0,MAO,MADGOAN JN.,12:10:00,00:00:00,78.0,SWV,SAWANTWADI ROAD,MAO,MADGOAN JN.
4,108,VLNK-MAO-SWV,1.0,MAO,MADGOAN JN.,00:00:00,20:30:00,0.0,MAO,MADGOAN JN.,SWV,SAWANTWADI ROAD


In [7]:

print("Processed files:")
for path in (PROJECT_DIR / "data" / "processed").glob("*"):
    print(path.name)


Processed files:
railway_network_edges.csv
railway_network_edges_flagged.csv
railway_timetable_clean.csv


In [8]:

# Summarize each train's timetable route
timetable_summary = (
    timetable_clean
    .sort_values(["Train No", "SEQ"])
    .groupby("Train No", as_index=False)
    .agg(
        timetable_start=("Station Code", "first"),
        timetable_end=("Station Code", "last"),
        timetable_distance_km=("Distance", "max"),
        timetable_stations=("Station Code", "nunique"),
    )
)

timetable_summary["Train No"] = (
    timetable_summary["Train No"].astype("string").str.strip()
)

# Read only the two required Kaggle columns in chunks
chunks = []

for chunk in pd.read_csv(
    KAGGLE_PATH,
    usecols=["train_number", "distance_km"],
    dtype={"train_number": "string"},
    chunksize=100_000,
):
    chunk["train_number"] = chunk["train_number"].str.strip()
    chunks.append(chunk)

kaggle_distances = pd.concat(chunks, ignore_index=True)
kaggle_distances["distance_km"] = pd.to_numeric(
    kaggle_distances["distance_km"], errors="coerce"
)

kaggle_summary = (
    kaggle_distances
    .groupby("train_number", as_index=False)
    .agg(
        kaggle_median_distance_km=("distance_km", "median"),
        kaggle_records=("distance_km", "size"),
    )
)

comparison = kaggle_summary.merge(
    timetable_summary,
    left_on="train_number",
    right_on="Train No",
    how="inner",
)

comparison["absolute_distance_difference_km"] = (
    comparison["kaggle_median_distance_km"]
    - comparison["timetable_distance_km"]
).abs()

print("Matched train IDs compared:", len(comparison))
display(
    comparison["absolute_distance_difference_km"].describe()
)

display(
    comparison.sort_values("absolute_distance_difference_km")[
        [
            "train_number",
            "kaggle_median_distance_km",
            "timetable_distance_km",
            "timetable_stations",
            "absolute_distance_difference_km",
        ]
    ].head(15)
)

# Summarize each train's timetable route
timetable_summary = (
    timetable_clean
    .sort_values(["Train No", "SEQ"])
    .groupby("Train No", as_index=False)
    .agg(
        timetable_start=("Station Code", "first"),
        timetable_end=("Station Code", "last"),
        timetable_distance_km=("Distance", "max"),
        timetable_stations=("Station Code", "nunique"),
    )
)

timetable_summary["Train No"] = (
    timetable_summary["Train No"].astype("string").str.strip()
)

# Read only the two required Kaggle columns in chunks
chunks = []

for chunk in pd.read_csv(
    KAGGLE_PATH,
    usecols=["train_number", "distance_km"],
    dtype={"train_number": "string"},
    chunksize=100_000,
):
    chunk["train_number"] = chunk["train_number"].str.strip()
    chunks.append(chunk)

kaggle_distances = pd.concat(chunks, ignore_index=True)
kaggle_distances["distance_km"] = pd.to_numeric(
    kaggle_distances["distance_km"], errors="coerce"
)

kaggle_summary = (
    kaggle_distances
    .groupby("train_number", as_index=False)
    .agg(
        kaggle_median_distance_km=("distance_km", "median"),
        kaggle_records=("distance_km", "size"),
    )
)

comparison = kaggle_summary.merge(
    timetable_summary,
    left_on="train_number",
    right_on="Train No",
    how="inner",
)

comparison["absolute_distance_difference_km"] = (
    comparison["kaggle_median_distance_km"]
    - comparison["timetable_distance_km"]
).abs()

print("Matched train IDs compared:", len(comparison))
display(
    comparison["absolute_distance_difference_km"].describe()
)

display(
    comparison.sort_values("absolute_distance_difference_km")[
        [
            "train_number",
            "kaggle_median_distance_km",
            "timetable_distance_km",
            "timetable_stations",
            "absolute_distance_difference_km",
        ]
    ].head(15)
)


Matched train IDs compared: 1820


count    1820.000000
mean      582.236264
std       601.944110
min         0.000000
25%       125.875000
50%       355.250000
75%       903.875000
max      3157.000000
Name: absolute_distance_difference_km, dtype: float64

,train_number,kaggle_median_distance_km,timetable_distance_km,timetable_stations,absolute_distance_difference_km
141,12183,754.0,754.0,11,0.0
1519,51606,253.0,253.0,28,0.0
392,12456,789.5,790.0,21,0.5
1080,22309,559.0,560.0,8,1.0
391,12455,792.0,790.0,21,2.0
1140,22462,651.5,654.0,8,2.5
1015,22112,601.0,604.0,20,3.0
663,12759,793.0,797.0,17,4.0
181,12225,800.0,795.0,15,5.0
1743,74909,147.0,152.0,19,5.0


Matched train IDs compared: 1820


count    1820.000000
mean      582.236264
std       601.944110
min         0.000000
25%       125.875000
50%       355.250000
75%       903.875000
max      3157.000000
Name: absolute_distance_difference_km, dtype: float64

,train_number,kaggle_median_distance_km,timetable_distance_km,timetable_stations,absolute_distance_difference_km
141,12183,754.0,754.0,11,0.0
1519,51606,253.0,253.0,28,0.0
392,12456,789.5,790.0,21,0.5
1080,22309,559.0,560.0,8,1.0
391,12455,792.0,790.0,21,2.0
1140,22462,651.5,654.0,8,2.5
1015,22112,601.0,604.0,20,3.0
663,12759,793.0,797.0,17,4.0
181,12225,800.0,795.0,15,5.0
1743,74909,147.0,152.0,19,5.0


In [9]:

# Compare the two distance values using absolute difference
thresholds = [10, 25, 50, 100, 200, 500]

total = len(comparison)

print("Total matched train IDs:", total)
print()

for threshold in thresholds:
    count = (
        comparison["absolute_distance_difference_km"] <= threshold
    ).sum()

    percentage = count / total * 100 if total else 0

    print(
        f"Difference <= {threshold:>3} km: "
        f"{count:>4} trains ({percentage:.1f}%)"
    )

# Inspect the largest discrepancies
print("\nLargest distance differences:")
display(
    comparison.nlargest(
        10, "absolute_distance_difference_km"
    )[
        [
            "train_number",
            "kaggle_median_distance_km",
            "timetable_distance_km",
            "timetable_stations",
            "absolute_distance_difference_km",
        ]
    ]
)


Total matched train IDs: 1820

Difference <=  10 km:   20 trains (1.1%)
Difference <=  25 km:   65 trains (3.6%)
Difference <=  50 km:  163 trains (9.0%)
Difference <= 100 km:  352 trains (19.3%)
Difference <= 200 km:  670 trains (36.8%)
Difference <= 500 km: 1088 trains (59.8%)

Largest distance differences:


,train_number,kaggle_median_distance_km,timetable_distance_km,timetable_stations,absolute_distance_difference_km
450,12516,773.0,3930.0,56,3157.0
449,12515,795.0,3932.0,57,3137.0
442,12508,811.5,3930.0,56,3118.5
441,12507,815.0,3932.0,58,3117.0
1158,22501,521.0,3617.0,41,3096.0
1159,22502,604.5,3617.0,39,3012.5
456,12522,764.0,3435.0,62,2671.0
1814,82651,572.0,3211.0,22,2639.0
455,12521,793.5,3427.0,63,2633.5
1815,82652,660.0,3212.0,22,2552.0
